In [ ]:
# ---- 1. Paths -------------------------------------------------------------
import os, sys, glob, shutil
from pathlib import Path

def _first(cands):
    for c in cands:
        for hit in glob.glob(c):
            p = Path(hit)
            if p.exists(): return p
    return None

DATA = _first(['/kaggle/input/birdclef-2026',
               '/kaggle/input/competitions/birdclef-2026',
               '/kaggle/input/*birdclef*2026*'])
assert DATA is not None and (DATA/'taxonomy.csv').exists(), 'birdclef-2026 not attached'

CODE = _first(['/kaggle/input/birdclef2026-code',
               '/kaggle/input/datasets/*/birdclef2026-code',
               '/kaggle/input/*/birdclef2026-code',
               '/kaggle/input/*birdclef2026-code*'])
assert CODE is not None, 'birdclef2026-code not attached'

def _resolve(parent, name, marker):
    for cand in (parent/name/name, parent/name):
        if (cand/marker).exists(): return cand
    raise FileNotFoundError(f'{name}/{marker} under {parent}')

SRC = _resolve(CODE, 'src', 'taxonomy.py')
DP  = _resolve(CODE, 'data_prep', 'make_folds.py')

# Locate Perch v2 SavedModel (any saved_model.pb under /kaggle/input)
PERCH = None
for hit in sorted(glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)):
    p = Path(hit).parent
    # require dir name or parent to contain 'bird' or 'perch' so we don't pick a random TF model
    s = str(p).lower()
    if 'bird' in s or 'perch' in s or 'vocaliz' in s:
        PERCH = p; break
if PERCH is None:
    # fallback: any saved_model.pb
    pbs = sorted(glob.glob('/kaggle/input/**/saved_model.pb', recursive=True))
    PERCH = Path(pbs[0]).parent if pbs else None
assert PERCH is not None, 'no SavedModel found - attach Perch v2'

os.environ['BIRDCLEF_DATA'] = str(DATA)
WORK = Path('/kaggle/working')
for s, n in ((SRC,'src'), (DP,'data_prep')):
    d = WORK/n
    if d.exists(): shutil.rmtree(d)
    shutil.copytree(s, d)
for m in list(sys.modules):
    if m.split('.')[0] in ('src','data_prep'): del sys.modules[m]
sys.path[:] = [str(WORK)] + [p for p in sys.path if p != str(WORK)]

print('data  :', DATA)
print('code  :', CODE)
print('perch :', PERCH)

In [ ]:
# ---- 2. Build folds + index ----------------------------------------------
import numpy as np, pandas as pd
from src.config import (TRAIN_AUDIO_DIR, TRAIN_SOUNDSCAPES_DIR, TRAIN_CSV,
                        TRAIN_SS_LABELS_CSV, CLIP_SAMPLES, SAMPLE_RATE)
from src.taxonomy import num_classes, class_to_idx
from data_prep.make_folds import build_folds

NC = num_classes()
C2I = class_to_idx()

FOLDS_CSV = WORK/'folds.csv'
fdf = build_folds(n_folds=5, seed=42)
fdf.to_csv(FOLDS_CSV, index=False)
fold_map = fdf.set_index('filename')['fold'].to_dict()

train_df = pd.read_csv(TRAIN_CSV)
ss_df    = pd.read_csv(TRAIN_SS_LABELS_CSV)
print(f'focal rows: {len(train_df)}   ss segments: {len(ss_df)}   classes: {NC}')
print('fold seg sums:')
seg_per = ss_df.groupby('filename').size()
print(fdf.assign(n=fdf['filename'].map(seg_per)).groupby('fold')['n'].sum().to_string())

In [ ]:
# ---- 3. Load Perch v2 (TF SavedModel) -------------------------------------
import tensorflow as tf
print('TF', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))

_perch = tf.saved_model.load(str(PERCH))
_sig = _perch.signatures['serving_default']
print('signature inputs :', {k: v.shape for k,v in _sig.structured_input_signature[1].items()})
print('signature outputs:', {k: v.shape for k,v in _sig.structured_outputs.items()})

# Probe with a 5s zero-vector to discover the embedding output name + dim
_in_key = list(_sig.structured_input_signature[1].keys())[0]
_probe = _sig(**{_in_key: tf.zeros((1, CLIP_SAMPLES), dtype=tf.float32)})
_cands = []
for k, v in _probe.items():
    arr = v.numpy() if hasattr(v, 'numpy') else np.asarray(v)
    if arr.ndim == 2 and arr.shape[0] == 1 and arr.shape[1] >= 256:
        _cands.append((k, arr.shape[1]))
_cands.sort(key=lambda kv: (-(kv[1]==1280), -kv[1]))  # prefer 1280, else largest
assert _cands, f'no embedding-shaped output found in {list(_probe.keys())}'
EMB_KEY, EMB_DIM = _cands[0]
IN_KEY = _in_key
print(f'using input "{IN_KEY}" -> output "{EMB_KEY}"  (dim={EMB_DIM})')

@tf.function(input_signature=[tf.TensorSpec(shape=[None, CLIP_SAMPLES], dtype=tf.float32)])
def perch_embed(batch):
    out = _sig(**{IN_KEY: batch})
    return out[EMB_KEY]

# Warmup
_ = perch_embed(tf.zeros((4, CLIP_SAMPLES), dtype=tf.float32)).numpy()
print('embedder ready')

In [ ]:
# ---- 4. Extract embeddings ------------------------------------------------
# Strategy: 1 center-energy crop per focal clip, all labeled ss segments.
# Save to /kaggle/working/cache/emb/<sid>.npy (float16) + emb_index.parquet
from src.audio import load_audio, topk_energy_crops
from tqdm.auto import tqdm
import time

EMB_DIR = WORK/'cache'/'emb'; EMB_DIR.mkdir(parents=True, exist_ok=True)
EMB_INDEX = WORK/'cache'/'emb_index.parquet'
BATCH = 32

def _to_sec(v):
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    if ':' in s:
        parts = [float(p) for p in s.split(':')]
        while len(parts) < 3: parts.insert(0, 0.0)
        h, m, sec = parts[-3], parts[-2], parts[-1]
        return h*3600.0 + m*60.0 + sec
    return float(s)

def _ensure(y, n=CLIP_SAMPLES):
    if len(y) < n: return np.pad(y, (0, n-len(y)))
    return y[:n] if len(y) > n else y

def _flush(buf_waves, buf_sids):
    if not buf_waves: return
    arr = np.stack(buf_waves).astype(np.float32)
    embs = perch_embed(tf.constant(arr)).numpy().astype(np.float16)
    for sid, e in zip(buf_sids, embs):
        np.save(EMB_DIR/f'{sid}.npy', e)

rows = []
buf_w, buf_s = [], []

# 4a. Focal: 1 top-energy crop per file
t0 = time.time()
for rec in tqdm(train_df.to_dict('records'), desc='focal'):
    fpath = TRAIN_AUDIO_DIR/rec['filename']
    sid = f"focal_{Path(rec['filename']).stem.replace('/', '_')}_0"
    out = EMB_DIR/f'{sid}.npy'
    if not out.exists():
        try: y = load_audio(fpath)
        except Exception: continue
        crop = topk_energy_crops(y, k=1)[0]
        buf_w.append(_ensure(crop)); buf_s.append(sid)
        if len(buf_w) >= BATCH:
            _flush(buf_w, buf_s); buf_w, buf_s = [], []
    rows.append({
        'sample_id': sid, 'source':'focal', 'file': rec['filename'],
        'seg_start': -1.0, 'seg_end': -1.0,
        'labels': rec['primary_label'],
        'secondary_labels': rec.get('secondary_labels','') or '',
        'weight_group':'primary', 'rating': float(rec.get('rating', 0.0) or 0.0),
        'fold': -1,
    })
_flush(buf_w, buf_s); buf_w, buf_s = [], []
print(f'focal done in {(time.time()-t0)/60:.1f} min')

# 4b. Soundscapes
t0 = time.time()
for rec in tqdm(ss_df.to_dict('records'), desc='soundscape'):
    fpath = TRAIN_SOUNDSCAPES_DIR/rec['filename']
    stem = Path(rec['filename']).stem
    start_sec = _to_sec(rec['start'])
    end_sec = _to_sec(rec.get('end', start_sec + 5.0))
    sid = f"ss_{stem}_{int(start_sec):03d}"
    out = EMB_DIR/f'{sid}.npy'
    if not out.exists():
        try: y = load_audio(fpath)
        except Exception: continue
        start = int(start_sec * SAMPLE_RATE)
        seg = _ensure(y[start:start+CLIP_SAMPLES])
        buf_w.append(seg); buf_s.append(sid)
        if len(buf_w) >= BATCH:
            _flush(buf_w, buf_s); buf_w, buf_s = [], []
    rows.append({
        'sample_id': sid, 'source':'soundscape', 'file': rec['filename'],
        'seg_start': start_sec, 'seg_end': end_sec,
        'labels': rec['primary_label'], 'secondary_labels':'',
        'weight_group':'soundscape', 'rating': 5.0,
        'fold': int(fold_map.get(rec['filename'], 0)),
    })
_flush(buf_w, buf_s)
print(f'ss done in {(time.time()-t0)/60:.1f} min')

idx_df = pd.DataFrame(rows)
idx_df.to_parquet(EMB_INDEX, index=False)
print(f'index: {len(idx_df)} rows  ({idx_df.groupby("source").size().to_dict()})')
n_files = len(list(EMB_DIR.glob('*.npy')))
print(f'cached files: {n_files}')

In [ ]:
# ---- 5. MLP head dataset + model -----------------------------------------
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.metrics import birdclef_roc_auc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

def encode_targets(primary, secondary, sec_w=0.3):
    y = np.zeros(NC, dtype=np.float32)
    if primary in C2I: y[C2I[primary]] = 1.0
    for s in str(secondary or '').replace(',', ';').replace("'", '').replace('[','').replace(']','').split(';'):
        s = s.strip()
        if s and s in C2I: y[C2I[s]] = max(y[C2I[s]], sec_w)
    return y

class EmbDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        # preload embeddings into RAM (float16, ~30k * 1280 * 2B = 75 MB)
        self.embs = np.stack([np.load(EMB_DIR/f"{sid}.npy") for sid in self.df['sample_id']]).astype(np.float32)
        self.tgts = np.stack([encode_targets(r['labels'], r['secondary_labels']) for _,r in self.df.iterrows()])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        return {'x': torch.from_numpy(self.embs[i]), 't': torch.from_numpy(self.tgts[i])}

VAL_FOLD = 0
valid_sids = set(p.stem for p in EMB_DIR.glob('*.npy'))
idx_df = idx_df[idx_df['sample_id'].isin(valid_sids)].reset_index(drop=True)
val_mask = (idx_df['source']=='soundscape') & (idx_df['fold']==VAL_FOLD)
train_ds = EmbDS(idx_df[~val_mask])
val_ds   = EmbDS(idx_df[val_mask])
print(f'train={len(train_ds)}  val={len(val_ds)}  emb_dim={EMB_DIM}')

class MLPHead(nn.Module):
    def __init__(self, d_in, n_classes, hidden=512, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x): return self.net(x)

model = MLPHead(EMB_DIM, NC, hidden=512, drop=0.3).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params/1e6:.2f} M')

In [ ]:
# ---- 6. Train MLP head ----------------------------------------------------
# Distribution-mismatch fix: train is ~95% focal but val is 100% soundscape.
# Without resampling, the head overfits to focal style and val AUC collapses
# after epoch 1. Solution: oversample soundscape so each batch is ~50/50.
import math, time, json
from torch.utils.data import WeightedRandomSampler

EPOCHS = 12
BS = 512
LR = 3e-4
WD = 1e-4
MIXUP_A = 0.2
WARMUP_EPS = 1
SS_BATCH_FRAC = 0.5         # target fraction of soundscape rows per batch
EARLY_STOP_PATIENCE = 3

# Per-sample weights: balance focal vs soundscape
src_arr = train_ds.df['source'].values
n_focal = int((src_arr == 'focal').sum())
n_ss    = int((src_arr == 'soundscape').sum())
w_focal = (1 - SS_BATCH_FRAC) / max(1, n_focal)
w_ss    = SS_BATCH_FRAC      / max(1, n_ss)
sample_w = np.where(src_arr == 'soundscape', w_ss, w_focal).astype(np.float64)
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_ds), replacement=True)
print(f'sampler: focal={n_focal} (w={w_focal:.2e})  ss={n_ss} (w={w_ss:.2e})')

train_loader = DataLoader(train_ds, batch_size=BS, sampler=sampler, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BS*2, shuffle=False, num_workers=2,
                          pin_memory=True)

# Plain BCE — Perch embeddings are already class-discriminative, pos_weight
# combined with focal-dominated train set was destabilizing.
criterion = nn.BCEWithLogitsLoss()

optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
steps_per_ep = max(1, len(train_loader))
total_steps = EPOCHS * steps_per_ep
warmup_steps = WARMUP_EPS * steps_per_ep
def lr_lambda(s):
    if s < warmup_steps: return (s+1) / max(1, warmup_steps)
    p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))
sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

def mixup(x, t, alpha=MIXUP_A):
    if alpha <= 0: return x, t
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[perm], lam * t + (1-lam) * t[perm]

@torch.no_grad()
def evaluate():
    model.eval(); ps, ts = [], []
    for b in val_loader:
        p = torch.sigmoid(model(b['x'].to(device))).cpu().numpy()
        ps.append(p); ts.append(b['t'].numpy())
    P = np.concatenate(ps); T = (np.concatenate(ts) > 0.5).astype(np.float32)
    return birdclef_roc_auc(T, P)

best = -1.0; hist = []
OUT = WORK/'best_emb.pt'
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); losses = []
    for b in train_loader:
        x = b['x'].to(device, non_blocking=True)
        t = b['t'].to(device, non_blocking=True)
        x, t = mixup(x, t)
        optim.zero_grad(set_to_none=True)
        loss = criterion(model(x), t)
        loss.backward(); optim.step(); sched.step()
        losses.append(loss.item())
    auc = evaluate()
    lr_now = optim.param_groups[0]['lr']
    print(f'ep {ep+1:02d}/{EPOCHS}  loss={np.mean(losses):.4f}  val_auc={auc:.4f}  lr={lr_now:.2e}  ({time.time()-t0:.1f}s)')
    hist.append({'epoch': ep+1, 'loss': float(np.mean(losses)), 'auc': float(auc), 'lr': float(lr_now)})
    if auc > best:
        best = auc
        torch.save({
            'model': model.state_dict(),
            'cfg': {'kind':'mlphead', 'emb_dim': EMB_DIM, 'hidden': 512, 'drop': 0.3,
                    'embedder': 'perch_v2', 'in_key': IN_KEY, 'out_key': EMB_KEY,
                    'sample_rate': SAMPLE_RATE, 'clip_samples': CLIP_SAMPLES},
            'auc': float(auc), 'epoch': ep+1, 'num_classes': NC,
        }, OUT)
(WORK/'emb_train_history.json').write_text(json.dumps(hist, indent=2))
print(f'\ndone. best_val_auc={best:.4f}  saved -> {OUT}')

In [ ]:
# ---- 7. Quick full-soundscape AUC on saved best ---------------------------
ck = torch.load(OUT, map_location=device, weights_only=False)
model.load_state_dict(ck['model']); model.eval()

ss_full_df = idx_df[idx_df['source']=='soundscape'].reset_index(drop=True)
ss_ds = EmbDS(ss_full_df)
ss_loader = DataLoader(ss_ds, batch_size=BS*2, shuffle=False, num_workers=2)
ps, ts = [], []
with torch.no_grad():
    for b in ss_loader:
        ps.append(torch.sigmoid(model(b['x'].to(device))).cpu().numpy())
        ts.append(b['t'].numpy())
P = np.concatenate(ps); T = (np.concatenate(ts) > 0.5).astype(np.float32)
print(f'Model B full-soundscape macro AUC: {birdclef_roc_auc(T, P):.4f}  (N={len(P)})')
for f in sorted(ss_full_df["fold"].unique()):
    m = (ss_full_df['fold']==f).values
    if m.sum() < 20: continue
    print(f'  fold {f}: N={m.sum():<4} AUC={birdclef_roc_auc(T[m], P[m]):.4f}')
print('\nfiles in /kaggle/working/:')
for p in sorted(WORK.glob('*.pt')) + sorted(WORK.glob('*.json')):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.2f} MB)')